## 데이터 불러오기

파일도 작고 금방일거라 GPU로 바꾸면 금방 돌아갈거예요!

In [1]:
import pandas as pd

df=pd.read_csv('파일명')

FileNotFoundError: [Errno 2] No such file or directory: '파일명'

## 모델 불러오기

df의 content 행만 넣으므로 content만이 존재 하지 않을 시 합쳐주는 작업 필요

In [ ]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import torch

# Check for GPU availability
if torch.cuda.is_available():
    device = 0  # Use the first GPU
    print("GPU is available. Using GPU for classification.")
else:
    device = -1  # Use CPU
    print("GPU is not available. Using CPU for classification.")

# 광고/스팸 분류 모델
classifier = pipeline(
    "text-classification",
    model="blockenters/sms-spam-classifier",
    truncation=True,
    device=device
)

texts = df["content"].fillna("").tolist()

# 예측
results = classifier(texts, batch_size=32)

# 결과 컬럼 추가
df["label"] = [r["label"] for r in results]
df["score"] = [r["score"] for r in results]

# spam / not spam 분리
spam_df = df[df["label"] == "LABEL_1"].copy()
not_spam_df = df[df["label"] == "LABEL_0"].copy()

print("spam 개수:", len(spam_df))
print("not spam 개수:", len(not_spam_df))

In [ ]:
spam_df

In [ ]:
not_spam_df

In [ ]:
spam_df.to_csv('spam_data.csv', index=False, encoding='utf-8-sig')
print("spam_df saved to spam_data.csv")

In [ ]:
not_spam_df.to_csv('not_spam_data.csv', index=False, encoding='utf-8-sig')
print("not_spam_df saved to not_spam_data.csv")

이후 해당 두개 파일 확인하며 스팸이 제대로 분리 되었는지 확인